In [1]:
#!/usr/bin/env python
# coding: utf-8
import json
import os
import time
import uuid

In [2]:
import evaluate
import numpy as np
import torch
from datasets import load_from_disk
from peft import LoraConfig, PeftConfig, get_peft_model
from tqdm import tqdm
from unsloth import FastLanguageModel
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

from trl import SFTTrainer
from unsloth import is_bfloat16_supported

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


!pip install peft -U

# Supervised Fine Tuning

In [3]:
tqdm.pandas()

In [4]:
from pathlib import Path

In [5]:
import pandas as pd

In [6]:
TRAIN_BATCH_SIZE = 25
EVALUATION_BATCH_SIZE = 10
LEARNING_RATE = 5e-5  # 1e-3
LORA_PARAM_R = 16
LORA_PARAM_ALPHA = 16
LORA_PARAM_TARGET_MODULES = {
    "bigscience/mt0-small": ["q", "v"],
    "microsoft/phi-1_5": [
        "q_proj",
        "k_proj",
        "v_proj",
    ],
    "microsoft/Phi-3-mini-4k-instruct": ["qkv_proj"],
    "microsoft/Phi-3-medium-4k-instruct": ["qkv_proj"],
    "unsloth/Qwen2-7B-Instruct-bnb-4bit": [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
}
# PRECISION = torch.float32
PRECISION_NAME = 'bfloat16'
DEVICE = "cuda"  # 0 if torch.cuda.is_available() else "cpu"
CHOSEN_MODEL = "unsloth/Qwen2-7B-Instruct-bnb-4bit"  # "microsoft/phi-1_5"  "bigscience/mt0-small" "google/flan-t5-large"
TESTING = False
RUN_ID = uuid.uuid4().hex
print(RUN_ID)

ae517069e3734bb4884dfa7fed5db18f


In [7]:
print(f"Model: {CHOSEN_MODEL} will be trained on device: {DEVICE}.")

Model: unsloth/Qwen2-7B-Instruct-bnb-4bit will be trained on device: cuda.


### Developed utility functions<br>
- For details see [Bath github link](https://github.bath.ac.uk/gt566/ai-msc-dissertation/blob/dissertation-experienced-ft/nyx/dissertation/utils.py)

In [8]:
import sys
from pathlib import Path

path = Path.cwd().parent.absolute()
nyx_path = f'{path}/'
print(nyx_path)
sys.path.append(nyx_path)

/home/ubuntu/demerzel/


In [9]:
from nyx.constants import (
    COMMON_OUTPUT_PATHS,
    METRICS_PATH,
    SFT_DATA_OUTPUT_PATH,
    SFT_OUTPUT_DIR,
    SFT_PEFT_ADAPTER_PATH,
    SFT_PEFT_MERGED_MODEL_PATH,
)
from nyx.utils import (
    download_and_save_reddit_data,
    get_task_type,
    precision_enumerator,
    print_number_of_trainable_model_parameters,
    round_dictionary_values,
)
from nyx.evaluation import quantitative_comparison
from nyx.data_generation.prompts.model_specific_tokens import (
    QWEN_EOS,
    QWEN_BOS_USER,
    QWEN_BOS_ASSISTANT,
)

In [10]:
COMMON_OUTPUT_PATHS = COMMON_OUTPUT_PATHS.format(RUN_ID=RUN_ID)
METRICS_PATH = METRICS_PATH.format(COMMON_OUTPUT_PATHS=COMMON_OUTPUT_PATHS)
COMMON_OUTPUT_PATHS = COMMON_OUTPUT_PATHS.format(RUN_ID=RUN_ID)
SFT_OUTPUT_DIR = SFT_OUTPUT_DIR.format(COMMON_OUTPUT_PATHS=COMMON_OUTPUT_PATHS)
SFT_PEFT_ADAPTER_PATH = SFT_PEFT_ADAPTER_PATH.format(
    COMMON_OUTPUT_PATHS=COMMON_OUTPUT_PATHS
)
SFT_PEFT_MERGED_MODEL_PATH = SFT_PEFT_MERGED_MODEL_PATH.format(
    COMMON_OUTPUT_PATHS=COMMON_OUTPUT_PATHS
)

In [11]:
PRECISION = precision_enumerator(PRECISION_NAME)
PRECISION

torch.bfloat16

## Load model and data

In [12]:
# References
# ----------
# https://colab.research.google.com/drive/1W0j3rP8WpgxRdUgkb5l6E00EEVyjEZGk?usp=sharing#scrollTo=QmUBVEnvCDJv


max_seq_length = 4096  # Choose any! We auto support RoPE Scaling internally!
dtype = (
    None  # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
)
load_in_4bit = True  # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
# Try more models at https://huggingface.co/unsloth!

original_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CHOSEN_MODEL,  # Reminder we support ANY Hugging Face model!
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)
tokenizer.padding_side = 'left'

==((====))==  Unsloth 2024.8: Fast Qwen2 patching. Transformers = 4.43.4.
   \\   /|    GPU: NVIDIA A10. Max memory: 21.988 GB. Platform = Linux.
O^O/ \_/ \    Pytorch: 2.3.0+cu121. CUDA = 8.6. CUDA Toolkit = 12.1.
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.26.post1. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth


/home/ubuntu/miniconda3/envs/empire/lib/python3.10/site-packages/transformers/quantizers/auto.py:174: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)
<string>:209: DeprecationWarning: invalid escape sequence '\ '
<string>:210: DeprecationWarning: invalid escape sequence '\_'
<string>:211: DeprecationWarning: invalid escape sequence '\ '
<string>:209: DeprecationWarning: invalid escape sequence '\ '
<string>:210: DeprecationWarning: invalid escape sequence '\_'
<string>:211: DeprecationWarning: invalid escape sequence '\ '


In [13]:
print(print_number_of_trainable_model_parameters(original_model))

trainable model parameters: 545201664
all model parameters: 4352972288
percentage of trainable model parameters: 12.52%


In [14]:
filtered_reddit_summarisation_data = Path(SFT_DATA_OUTPUT_PATH)
if not filtered_reddit_summarisation_data.is_dir():
    print("Downloading and saving filtered reddit data.")
    download_and_save_reddit_data()

In [15]:
dataset = load_from_disk(SFT_DATA_OUTPUT_PATH)
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'subreddit', 'title', 'post', 'summary'],
        num_rows: 116722
    })
    test: Dataset({
        features: ['id', 'subreddit', 'title', 'post', 'summary'],
        num_rows: 6553
    })
    validation: Dataset({
        features: ['id', 'subreddit', 'title', 'post', 'summary'],
        num_rows: 6447
    })
})

In [16]:
if TESTING is True:
    dataset["train"] = dataset["train"].select(range(200))
    dataset["test"] = dataset["test"].select(range(200))
    dataset["validation"] = dataset["validation"].select(range(50))
else:
    dataset = dataset.filter(lambda example, index: index % 10 == 0, with_indices=True)
dataset

Filter:   0%|          | 0/116722 [00:00<?, ? examples/s]

Filter:   0%|          | 0/6553 [00:00<?, ? examples/s]

Filter:   0%|          | 0/6447 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'subreddit', 'title', 'post', 'summary'],
        num_rows: 11673
    })
    test: Dataset({
        features: ['id', 'subreddit', 'title', 'post', 'summary'],
        num_rows: 656
    })
    validation: Dataset({
        features: ['id', 'subreddit', 'title', 'post', 'summary'],
        num_rows: 645
    })
})

In [17]:
tokenizer.pad_token = (
    tokenizer.pad_token if tokenizer.pad_token is not None else tokenizer.eos_token
)

In [18]:
EOS_TOKEN = QWEN_EOS
BOS_USER_TOKEN = QWEN_BOS_USER
BOS_ASSISTANT_TOKEN = QWEN_BOS_ASSISTANT


def prompt_prep_func(example):
    start_prompt = f"{BOS_USER_TOKEN}\nSummarize the following reddit post:\n"
    end_prompt = "\n\nSummary: "
    prompt = [
        start_prompt
        + post
        + EOS_TOKEN
        + f'\n{BOS_ASSISTANT_TOKEN}\n{end_prompt}{summary}{EOS_TOKEN}'
        for post, summary in zip(example["post"], example['summary'])
    ]
    example['text'] = prompt
    # example["input_ids"] = tokenizer(
    #     prompt,
    #     padding="max_length",
    #     truncation=True,
    #     return_tensors="pt",  # padding=True
    # ).input_ids  # .to(torch.device(DEVICE))
    # example["labels"] = tokenizer(
    #     example["summary"],
    #     padding="max_length",
    #     truncation=True,
    #     return_tensors="pt",  # padding=True
    # ).input_ids  # .to(torch.device(DEVICE))
    return example

The dataset actually contains 3 diff splits: train, validation, test.<br>
The tokenize_function code is handling all data across all splits in batches.

In [19]:
sft_dataset = dataset.map(prompt_prep_func, batched=True)
sft_dataset = sft_dataset.remove_columns(
    [
        "id",
        "subreddit",
        "post",
        "summary",
    ]
)

Map:   0%|          | 0/11673 [00:00<?, ? examples/s]

Map:   0%|          | 0/656 [00:00<?, ? examples/s]

Map:   0%|          | 0/645 [00:00<?, ? examples/s]

In [20]:
print(f"Shapes of the datasets:")
print(f"Training: {sft_dataset['train'].shape}")
print(f"Validation: {sft_dataset['validation'].shape}")
print(f"Test: {sft_dataset['test'].shape}")

Shapes of the datasets:
Training: (11673, 2)
Validation: (645, 2)
Test: (656, 2)


In [21]:
print(sft_dataset)

DatasetDict({
    train: Dataset({
        features: ['title', 'text'],
        num_rows: 11673
    })
    test: Dataset({
        features: ['title', 'text'],
        num_rows: 656
    })
    validation: Dataset({
        features: ['title', 'text'],
        num_rows: 645
    })
})


## Train PEFT adapter

Checking for layers to apply LoRA. Selecting the query and value layers are the most<br>
basic implementation according to the paper. They are refered to as q and v here.<br>
print(original_model)

In [22]:
# lora_config = LoraConfig(
#     # Determines the size of LoRA matrices. x*r * r*y = x*y
#     r=LORA_PARAM_R,
#     # scaling coefficient. Paper mentions it is important because the adjustments are small compared
#     # to the rest of the model.
#     lora_alpha=LORA_PARAM_ALPHA,
#     # Variable target_modules determines what layers are fine-tuned, see architecture above.
#     # Simplest case scenario based on the original paper.
#     target_modules=LORA_PARAM_TARGET_MODULES[CHOSEN_MODEL],
#     lora_dropout=0.05,
#     bias="none",
#     task_type=get_task_type(model=original_model),
# )
peft_model = FastLanguageModel.get_peft_model(
    original_model,
    r=LORA_PARAM_R,  # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules=LORA_PARAM_TARGET_MODULES[CHOSEN_MODEL],
    lora_alpha=LORA_PARAM_ALPHA,
    lora_dropout=0,  # Supports any, but = 0 is optimized
    bias="none",  # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing="unsloth",  # True or "unsloth" for very long context
    random_state=42,
    use_rslora=False,  # We support rank stabilized LoRA
    loftq_config=None,  # And LoftQ
)

Unsloth 2024.8 patched 28 layers with 0 QKV layers, 28 O layers and 28 MLP layers.


In [23]:
# peft_model = get_peft_model(original_model, lora_config)
print(print_number_of_trainable_model_parameters(peft_model))

trainable model parameters: 40370176
all model parameters: 4393342464
percentage of trainable model parameters: 0.92%


In [24]:
peft_training_args = TrainingArguments(
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=8,
    # Use num_train_epochs = 1, warmup_ratio for full training runs!
    warmup_steps=20,
    max_steps=len(sft_dataset["train"]) // TRAIN_BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=1,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=42,
    output_dir=SFT_OUTPUT_DIR,
)


trainer = SFTTrainer(
    model=peft_model,
    tokenizer=tokenizer,
    train_dataset=sft_dataset['train'],
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    args=peft_training_args,
)

Map (num_proc=2):   0%|          | 0/11673 [00:00<?, ? examples/s]

max_steps is given, it will override any value given in num_train_epochs


In [25]:
# peft_trainer = Trainer(
#     model=peft_model,  # Important to train on Mac Chip GPU equivalent # .to(
#     #     torch.device(DEVICE)
#     # )
#     args=peft_training_args,
#     train_dataset=tokenized_datasets["train"],
# )

# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA A10. Max memory = 21.988 GB.
5.764 GB of memory reserved.


In [26]:
start = time.time()
trainer_stats = trainer.train()
end = time.time()
duration = end - start
# print(end)
print(f"Training for 1 epoch took {round(duration, 2)} seconds to execute.")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 11,673 | Num Epochs = 9
O^O/ \_/ \    Batch size per device = 25 | Gradient Accumulation steps = 8
\        /    Total batch size = 200 | Total steps = 466
 "-____-"     Number of trainable parameters = 40,370,176


Step,Training Loss
1,2.637000
2,2.671600
3,2.597200
4,2.661900
5,2.644800
6,2.651700
7,2.595600
8,2.630600
9,2.634900
10,2.595300


Training for 1 epoch took 35795.4 seconds to execute.


In [ ]:
# @title Show final memory and time stats

used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

In [27]:
peft_model.save_pretrained(SFT_PEFT_ADAPTER_PATH)
print(f'Peft adapter was saved to:\n{SFT_PEFT_ADAPTER_PATH}')
tokenizer.save_pretrained(SFT_PEFT_ADAPTER_PATH)

# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

Peft adapter was saved to:
./experiments/ae517069e3734bb4884dfa7fed5db18f/models/supervised-fine-tuning/peft-checkpoint-local


('./experiments/ae517069e3734bb4884dfa7fed5db18f/models/supervised-fine-tuning/peft-checkpoint-local/tokenizer_config.json',
 './experiments/ae517069e3734bb4884dfa7fed5db18f/models/supervised-fine-tuning/peft-checkpoint-local/special_tokens_map.json',
 './experiments/ae517069e3734bb4884dfa7fed5db18f/models/supervised-fine-tuning/peft-checkpoint-local/vocab.json',
 './experiments/ae517069e3734bb4884dfa7fed5db18f/models/supervised-fine-tuning/peft-checkpoint-local/merges.txt',
 './experiments/ae517069e3734bb4884dfa7fed5db18f/models/supervised-fine-tuning/peft-checkpoint-local/added_tokens.json',
 './experiments/ae517069e3734bb4884dfa7fed5db18f/models/supervised-fine-tuning/peft-checkpoint-local/tokenizer.json')

## Load PEFT adapter

In [ ]:
# adapter_checkpoint_path = f"{common_folder_path}/peft-dialogue-summary-checkpoint-local"
# trained_model = AutoModelForSeq2SeqLM.from_pretrained(
#     CHOSEN_MODEL, torch_dtype=PRECISION
# )
# try:
#     trained_model = AutoModelForSeq2SeqLM.from_pretrained(<br>
#         CHOSEN_MODEL, torch_dtype=PRECISION
#     )
# except ValueError:
#     trained_model = AutoModelForCausalLM.from_pretrained(<br>
#         CHOSEN_MODEL, torch_dtype=PRECISION
#     )
# print("ok")

## _Comparing PEFT and Baseline model generations (with ROUGE)_

In [28]:
N_EVAL_SAMPLES = int(len(sft_dataset['test']) * 1.0)  # only using 10% of data
print(N_EVAL_SAMPLES)

656


In [48]:
FastLanguageModel.for_inference(original_model)


start = time.time()
baseline_model_generation = quantitative_comparison(
    original_model,
    dataset,
    tokenizer,
    n_samples_to_evaluate=N_EVAL_SAMPLES,
    batch_size=EVALUATION_BATCH_SIZE,
    device=DEVICE,
)
# peft_checkpoint_model = PeftModel.from_pretrained(original_model, SFT_PEFT_ADAPTER_PATH)

In [49]:
with torch.no_grad():
    torch.cuda.empty_cache()

In [51]:
# peft_config = PeftConfig.from_pretrained(SFT_PEFT_ADAPTER_PATH)
# # to initiate with random weights
# peft_config.init_lora_weights = False
# original_model.add_adapter(peft_config)
# original_model.enable_adapters()
# original_model  # .to(torch.device(DEVICE))


peft_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_PEFT_ADAPTER_PATH,  # YOUR MODEL YOU USED FOR TRAINING
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)
FastLanguageModel.for_inference(peft_model)  # Unsloth has 2x faster inference!

==((====))==  Unsloth 2024.8: Fast Qwen2 patching. Transformers = 4.43.4.
   \\   /|    GPU: NVIDIA A10. Max memory: 21.988 GB. Platform = Linux.
O^O/ \_/ \    Pytorch: 2.3.0+cu121. CUDA = 8.6. CUDA Toolkit = 12.1.
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.26.post1. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth


ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `load_in_8bit_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [ ]:
peft_checkpoint_generation = quantitative_comparison(
    peft_model,  # peft enabled model
    dataset,
    tokenizer,
    n_samples_to_evaluate=N_EVAL_SAMPLES,
    batch_size=EVALUATION_BATCH_SIZE,
    device=DEVICE,
)
end = time.time()
duration = end - start
print(
    f"Evaluating N={N_EVAL_SAMPLES} samples took {round(duration, 2)} seconds to execute."
)

In [ ]:
human_baseline_answer = dataset["test"][0:N_EVAL_SAMPLES]["summary"]

In [ ]:
zipped_summaries = list(
    zip(human_baseline_answer, peft_checkpoint_generation, baseline_model_generation)
)

In [ ]:
df = pd.DataFrame(
    zipped_summaries,
    columns=[
        "human_baseline_answer",
        "peft_checkpoint_generation",
        "baseline_model_generation",
    ],
)
df.head()
print(df.shape)

In [ ]:
rouge = evaluate.load("rouge")

In [ ]:
original_model_results = rouge.compute(
    predictions=baseline_model_generation,
    references=human_baseline_answer[0 : len(baseline_model_generation)],
    use_aggregator=True,
    use_stemmer=True,
)

In [ ]:
peft_model_results = rouge.compute(
    predictions=peft_checkpoint_generation,
    references=human_baseline_answer[0 : len(peft_checkpoint_generation)],
    use_aggregator=True,
    use_stemmer=True,
)

In [ ]:
original_model_results = round_dictionary_values(original_model_results)
# instruct_model_results = round_dictionary_values(instruct_model_results)
peft_model_results = round_dictionary_values(peft_model_results)
print("ORIGINAL MODEL:")
print(original_model_results)
# print('INSTRUCT MODEL:')
# print(instruct_model_results)
print("PEFT MODEL:")
print(peft_model_results)

print("Absolute percentage improvement of PEFT MODEL over ORIGINAL MODEL")
improvement = np.array(list(peft_model_results.values())) - np.array(
    list(original_model_results.values())
)
for key, value in zip(peft_model_results.keys(), improvement):
    print(f'{key}: {value * 100:.2f}%')

In [41]:
print(f'Metrics will be saved to path:\n{METRICS_PATH}')

Metrics will be saved to path:
./experiments/ae517069e3734bb4884dfa7fed5db18f/metrics


In [42]:
if not os.path.exists(METRICS_PATH):
    os.makedirs(METRICS_PATH)

data_path = f'{METRICS_PATH}/sft-results.json'

results_dict = {
    'rouge-metric-baseline-model': original_model_results,
    'rouge-metric-sft-model': peft_model_results,
    'train_batch_size': TRAIN_BATCH_SIZE,
    'evaluation_batch_size': EVALUATION_BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'lora_param_r': LORA_PARAM_R,
    'lora_param_alpha': LORA_PARAM_ALPHA,
    'lora_param_target_modules': LORA_PARAM_TARGET_MODULES[CHOSEN_MODEL],
    'precision': PRECISION_NAME,
    'device': DEVICE,
    'chosen_model': CHOSEN_MODEL,
    'testing': TESTING,
    'run_id': RUN_ID,
    'gpu_type': torch.cuda.get_device_name(),
}

In [43]:
with open(data_path, 'w') as file:
    json.dump(results_dict, file)

## Merge and save peft model (with base model)<br>
So that, it can be loaded in as a Reward Moldel.

In [44]:
peft_model = peft_model.merge_and_unload()
peft_model.save_pretrained(SFT_PEFT_MERGED_MODEL_PATH)

/home/ubuntu/miniconda3/envs/empire/lib/python3.10/site-packages/peft/tuners/lora/bnb.py:336: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


# END